In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Pre-processing of dataset

The FLAIR dataset provides centroid positions of patches in a JSON-based
file (`flair-2_centroids_sp_to_patch.json`). For training our
super-resolution model, we need these centroids converted into a structured
JSON format specific to the domain we are using.

This conversion is essential because it allows us to correctly map LR and HR
patches based on their centroid locations when building the dataloader.
Accurate LR–HR alignment ensures that each training pair corresponds to the
same spatial region in the original FLAIR images.

In [ ]:
import os
import tifffile
import json
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

In [ ]:

json_path = "/kaggle/working/coords_mapping.json"

# Save dictionary to JSON
with open(json_path, "w") as f:
    json.dump(mapping, f, indent=4)

print(f"Mapping saved to {json_path}")

with open(json_path, "r") as f:
    mapping = json.load(f)

print(type(mapping))       
print(len(mapping))        
print(list(mapping.items())[:5])  # Print first 5 key-value pairs

* The FLAIR dataset provides spatially aligned LR (10×10) satellite patches and HR (512×512) aerial patches, indexed using centroid coordinates.

* Centroid metadata from flair-2_centroids_sp_to_patch.json is converted into a structured coords_mapping.json for consistent LR–HR correspondence.

* A 4×4 sliding window is applied over the coordinate grid to identify spatially adjacent sets of 16 patches.

* For each window, a 4×4 list of LR patch identifiers and the corresponding 4×4 list of HR patch identifiers are extracted.

* Sixteen LR patches (each 10×10) are arranged in a 4×4 layout forming a 40×40 LR mosaic.

* Sixteen HR patches (each 512×512) are first downsampled to 128×128 to match the LR scale factor.

* The downsampled HR patches are arranged in a 4×4 layout forming a 512×512 HR mosaic.

* The resulting LR (40×40) and HR (512×512) mosaics represent the same geographic region and constitute one aligned training pair for super-resolution.

* Repeating this sliding-window process yields a large dataset of spatially coherent LR–HR mosaic pairs for model training.

In [ ]:


def load_coordinates(json_path):
    """Load coordinates from JSON file"""
    with open(json_path, 'r') as f:
        coords_data = json.load(f)
    return coords_data

def create_image_grid(coords_data, aerial_dir, grid_size=4, stride=2):
    aerial_files = [f for f in os.listdir(aerial_dir) if f.endswith('.tif')]
    print(f"Found {len(aerial_files)} aerial images")

    image_data = []
    for img_file in aerial_files:
        for coord_entry in coords_data:
            if 'image' in coord_entry and coord_entry['image'] == img_file:
                x = coord_entry.get('x', coord_entry.get('longitude', 0))
                y = coord_entry.get('y', coord_entry.get('latitude', 0))
                image_data.append((img_file, x, y))
                break
            elif 'filename' in coord_entry and coord_entry['filename'] == img_file:
                coords = coord_entry.get('coordinates', coord_entry.get('coords', [0, 0]))
                if len(coords) >= 2:
                    x, y = coords[0], coords[1]
                    image_data.append((img_file, x, y))
                break

    print(f"Found coordinates for {len(image_data)} images")

    if not image_data:
        print("Trying alternative coordinate parsing...")
        image_data = try_alternative_parsing(coords_data, aerial_files)

    if not image_data:
        raise ValueError("No image coordinates found! Check JSON structure.")

    image_data.sort(key=lambda item: (item[2], item[1]))

    y_coords = sorted(list(set([item[2] for item in image_data])))
    x_coords = sorted(list(set([item[1] for item in image_data])))

    print(f"Unique X coordinates: {len(x_coords)}, Range: {min(x_coords):.2f} to {max(x_coords):.2f}")
    print(f"Unique Y coordinates: {len(y_coords)}, Range: {min(y_coords):.2f} to {max(y_coords):.2f}")

    coord_to_file = {(item[1], item[2]): item[0] for item in image_data}

    grid_list = []

    for i in range(0, len(y_coords) - grid_size + 1, stride):
        row_slice = y_coords[i:i + grid_size]

        for j in range(0, len(x_coords) - grid_size + 1, stride):
            col_slice = x_coords[j:j + grid_size]

            grid = []
            valid_grid = True

            for y in row_slice:
                row = []
                for x in col_slice:
                    if (x, y) in coord_to_file:
                        row.append(coord_to_file[(x, y)])
                    else:
                        valid_grid = False
                        break
                if not valid_grid:
                    break
                grid.append(row)

            if valid_grid and len(grid) == grid_size and all(len(row) == grid_size for row in grid):
                grid_list.append(grid)

    return grid_list, x_coords, y_coords

def try_alternative_parsing(coords_data, aerial_files):
    """Alternative coordinate parsing methods"""
    image_data = []

    if isinstance(coords_data, dict):
        for img_file in aerial_files:
            if img_file in coords_data:
                coords = coords_data[img_file]
                if isinstance(coords, list) and len(coords) >= 2:
                    x, y = coords[0], coords[1]
                    image_data.append((img_file, x, y))

    elif isinstance(coords_data, list) and len(coords_data) > 0:
        for item in coords_data:
            if isinstance(item, list) and len(item) >= 3:
                filename, x, y = item[0], item[1], item[2]
                if filename in aerial_files:
                    image_data.append((filename, x, y))

    return image_data

def print_first_grids(grid_list, num_grids=2):
    for grid_idx, grid in enumerate(grid_list[:num_grids]):
        print(f"\n--- Grid {grid_idx + 1} ---")
        for row_idx, row in enumerate(grid):
            print(f"Row {row_idx + 1}: {row}")

def main():
    coords_path = "/kaggle/working/coords_mapping.json"
    aerial_dir = "/kaggle/input/d036-2020/aerial/Z10_FN"

    print("Loading coordinates...")
    coords_data = load_coordinates(coords_path)
    print(f"JSON data type: {type(coords_data)}")

    if isinstance(coords_data, list):
        print(f"JSON list length: {len(coords_data)}")
        if len(coords_data) > 0:
            print(f"First item: {coords_data[0]}")
    elif isinstance(coords_data, dict):
        print(f"JSON dict keys: {list(coords_data.keys())[:5]}...")

    print("\nCreating 4x4 sliding grid structure...")
    grid_list, x_coords, y_coords = create_image_grid(coords_data, aerial_dir, grid_size=4, stride=1)

    print(f"\nGenerated {len(grid_list)} complete 4x4 grids")

    if grid_list:
        print("\nFirst 2 complete 4x4 grids:")
        print_first_grids(grid_list, num_grids=2)

        print(f"\nStatistics:")
        print(f"Total complete 4x4 grids: {len(grid_list)}")
        print(f"Total images in grids: {len(grid_list) * 16}")
        print(f"X coordinate range: {min(x_coords):.2f} to {max(x_coords):.2f}")
        print(f"Y coordinate range: {min(y_coords):.2f} to {max(y_coords):.2f}")
        print(f"Grid structure: {len(grid_list)} grids × 4 rows × 4 columns")
    else:
        print("No complete 4x4 grids found!")

    return grid_list, x_coords, y_coords

if __name__ == "__main__":
    grid_list, x_coords, y_coords = main()


In [ ]:
def save_aerial_mosaic(grid, aerial_dir, save_dir, counter=1, target_size=(128,128)):
   
    os.makedirs(save_dir, exist_ok=True)

    # Transpose the grid first
    transposed_grid = [list(row) for row in zip(*grid)]
    
    mosaic_rows = []
    for row in transposed_grid:
        row_images = []
        for img_name in row:
            img_path = os.path.join(aerial_dir, img_name)
            try:
                aerial_img = tifffile.imread(img_path)
                if aerial_img.ndim == 3 and aerial_img.shape[-1] >= 3:
                    rgb_img = aerial_img[:, :, :3]
                else:
                    rgb_img = aerial_img

                rgb_img = rgb_img.astype(np.float32)
                if rgb_img.max() > rgb_img.min():
                    rgb_img = (rgb_img - rgb_img.min()) / (rgb_img.max() - rgb_img.min())

                rgb_img_resized = resize(rgb_img, target_size, preserve_range=True, anti_aliasing=True)
                row_images.append(rgb_img_resized)
            except Exception:
                placeholder = np.zeros((target_size[0], target_size[1], 3), dtype=np.float32)
                row_images.append(placeholder)

        mosaic_row = np.concatenate(row_images, axis=1)
        mosaic_rows.append(mosaic_row)

    mosaic_image = np.concatenate(mosaic_rows, axis=0)
    save_path = os.path.join(save_dir, f"img{counter}.tif")
    tifffile.imwrite(save_path, mosaic_image.astype(np.float32))

    return counter + 1, save_path

In [ ]:
def save_satellite_mosaic_from_grid(grid, sat_path, coords_file, save_dir, counter=1, patch_radius=5):
   
    os.makedirs(save_dir, exist_ok=True)

    # Load satellite data
    sat_data = np.load(sat_path, allow_pickle=True)
    if sat_data.shape[0] == 0:
        raise ValueError("Satellite data is empty!")

    # Always use timestamp 0
    sat_img = sat_data[0]  
  
    sat_img = np.transpose(sat_img[[2, 1, 0], :, :], (1, 2, 0))  
    sat_img = sat_img.astype(np.float32)
    sat_img -= sat_img.min()
    sat_img /= (sat_img.max() + 1e-6)

    H, W, C = sat_img.shape

    # Load coordinates
    import json
    with open(coords_file, "r") as f:
        coords = json.load(f)

   
    rows, cols = len(grid), len(grid[0])
    mosaic = np.zeros((rows * patch_radius * 2,
                       cols * patch_radius * 2,
                       C), dtype=np.float32)

    # Fill mosaic with patches
    for i, row in enumerate(grid):
        for j, img_name in enumerate(row):
            if isinstance(img_name, list):
                img_name = img_name[0]

            if img_name not in coords:
                continue

            cx, cy = coords[img_name]
            x1 = max(cx - patch_radius, 0)
            x2 = min(cx + patch_radius, W)
            y1 = max(cy - patch_radius, 0)
            y2 = min(cy + patch_radius, H)

            patch = sat_img[y1:y2, x1:x2, :]
            ph, pw = patch.shape[:2]

            mosaic[i*patch_radius*2:i*patch_radius*2+ph,
                   j*patch_radius*2:j*patch_radius*2+pw, :] = patch

    # Save mosaic as .tif
    save_path = os.path.join(save_dir, f"img{counter}.tif")
    tifffile.imwrite(save_path, mosaic.astype(np.float32))

    return counter + 1, save_path

In [ ]:
base_dir = "/kaggle/working/dataset"
z17_dir = os.path.join(base_dir, "aerialpatches")
os.makedirs(z17_dir, exist_ok=True)
print(f"Created directories:\n- {z17_dir}\n- {z17_dir}")

In [ ]:
# Directories
aerial_dir = "/kaggle/input/d036-2020/aerial/Z9_FF"
aerial_save_dir = "/kaggle/working/dataset/aerialpatches"

sentinel_path = "/kaggle/input/d036-2020/sentinel/Z9_FF/SEN2_sp_D036_2020-Z9_FF_data.npy"
coords_file = "/kaggle/working/coords_mapping.json"
sentinel_save_dir = "/kaggle/working/dataset/sentinelpatches"

import os
os.makedirs(aerial_save_dir, exist_ok=True)
os.makedirs(sentinel_save_dir, exist_ok=True)

for idx, grid in enumerate(grid_list):
    counter = idx + 1  

    # Save aerial mosaic
    _, aerial_path = save_aerial_mosaic(
        grid, aerial_dir, aerial_save_dir, counter=counter
    )

    # Save corresponding Sentinel mosaic 
    _, sat_path = save_satellite_mosaic_from_grid(
        grid, sentinel_path, coords_file, sentinel_save_dir, counter=counter
    )

    print(f"Mosaic {idx}: Aerial → {aerial_path}, Sentinel → {sat_path}")


In [ ]:

aerial_save_dir = "/kaggle/working/dataset/aerialpatches"

image_files = [f for f in os.listdir(aerial_save_dir) if f.lower().endswith(('.tif', '.tiff', '.png', '.jpg'))]

# Natural sort (so IMG_1, IMG_2 ... in correct order)
def natural_key(s):
    return [int(text) if text.isdigit() else text.lower() for text in re.split(r'(\d+)', s)]

image_files = sorted(image_files, key=natural_key)

# Function to map (row, col) → transposed index
def transpose_index(row, col, n=4):
    return col * n + row + 1  # 1-based index

# Rename files
n = 7 
for idx, old_name in enumerate(image_files):
    row, col = divmod(idx, n)              # row-major index
    new_idx = transpose_index(row, col, n) # transposed index
    ext = os.path.splitext(old_name)[1]    # keep extension
    new_name = f"IMG_{new_idx}{ext}"
    
    old_path = os.path.join(aerial_save_dir, old_name)
    new_path = os.path.join(aerial_save_dir, new_name)
    
    os.rename(old_path, new_path)
    print(f"Renamed {old_name} → {new_name}")


**To Visualise the aerial Images**

In [ ]:

# Path
aerial_dir = "/kaggle/working/dataset/aerialpatches"

# List all tif files
files = sorted([f for f in os.listdir(aerial_dir) if f.endswith(".tif")])
print(f"Found {len(files)} aerial images")

# Show first 12 images as a preview grid
n_show = min(49, len(files))
cols = 4
rows = (n_show + cols - 1) // cols

plt.figure(figsize=(16, 4 * rows))

for i, fname in enumerate(files[:n_show]):
    img_path = os.path.join(aerial_dir, fname)
    img = tifffile.imread(img_path)

    # Take only RGB channels if available
    if img.ndim == 3 and img.shape[-1] >= 3:
        img = img[:, :, :3]

    # Normalize for display
    img = img.astype("float32")
    if img.max() > 0:
        img /= img.max()

    plt.subplot(rows, cols, i + 1)
    plt.imshow(img)
    plt.title(fname, fontsize=8)
    plt.axis("off")

plt.tight_layout()
plt.show()


# **Per-Image and Grid-Level Labels**

1. Per-Image Labels
Extract unique class IDs from each aerial mask and save as image_labels.json:

"IMG_084438.tif": [0, 5, 6]


2. Grid-Level Aggregation
For each 4×4 mosaic grid:

Collect labels from all 16 patches

Compute the union of labels

Save as grid_combined_labels.json with keys like img1.tif

img1.tif → [0, 2, 5, 6, 7]


Purpose: Convert pixel-level mask annotations into mosaic-level labels for multi-label classification.

In [ ]:

aerial_dir = "/kaggle/input/d036-2020/aerial/Z9_FF"
mask_dir   = "/kaggle/input/d036-2020/labels/Z9_FF"

aerial_files = [f for f in os.listdir(aerial_dir) if f.endswith('.tif')]
image_labels_dict = {}

for img_file in aerial_files:
    base = os.path.splitext(img_file)[0]  
    mask_name = "MSK_" + base[4:] + ".tif"
    mask_path = os.path.join(mask_dir, mask_name)
    
    if os.path.exists(mask_path):
        mask = np.array(Image.open(mask_path))
        labels = np.unique(mask).tolist()  
        image_labels_dict[img_file] = labels
    else:
        image_labels_dict[img_file] = []


out_file = "/kaggle/working/image_labels.json"
with open(out_file, "w") as f:
    json.dump(image_labels_dict, f, indent=2)

print("Saved image-label dictionary to:", out_file)


In [ ]:
in_image_labels = "/kaggle/working/image_labels.json"
out_json = "/kaggle/working/grid_combined_labels.json"

if not os.path.exists(in_image_labels):
    raise FileNotFoundError(f"Per-image labels JSON not found at: {in_image_labels}")
if "grid_list" not in globals():
    raise NameError("grid_list not found. Make sure it is defined.")

# Load per-image labels
with open(in_image_labels, "r") as f:
    image_labels = json.load(f)

# Combine labels for each 4x4 grid
combined_dict = {}

for idx, grid in enumerate(grid_list, start=1):
    new_key = f"img{idx}.tif"
    combined_set = set()

    for row in grid:
        for imgname in row:
            labels = image_labels.get(imgname, [])
            try:
                label_ids = [int(x) for x in labels]
            except:
                try:
                    label_ids = [int(labels)]
                except:
                    label_ids = []
            combined_set.update(label_ids)

    combined_dict[new_key] = sorted(list(combined_set))

# Save combined labels
os.makedirs(os.path.dirname(out_json), exist_ok=True)
with open(out_json, "w") as f:
    json.dump(combined_dict, f, indent=2)

print(f"Saved combined labels for {len(combined_dict)} grids to: {out_json}")

# Print first 10 entries
for i, (k, v) in enumerate(combined_dict.items()):
    print(k, "->", v)
    if i >= 9:
        break


Transposing the labels according to grids 

In [ ]:

json_path = "/kaggle/working/grid_combined_labels.json"
output_path = "/kaggle/working/grid_combined_labels_transposed.json"


with open(json_path, "r") as f:
    data = json.load(f)

def natural_key(s): 
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r'(\d+)', s)]

keys = sorted(data.keys(), key=natural_key)
n = len(keys)
print(f"Total keys in dictionary: {n}")


n_cols = math.ceil(math.sqrt(n))
n_rows = math.ceil(n / n_cols)
print(f"Grid size: {n_rows} rows x {n_cols} cols")

grid = []
for i in range(0, n, n_cols):
    row = keys[i:i+n_cols]
    if len(row) < n_cols:
        row += [None]*(n_cols - len(row))  
    grid.append(row)

print("\nOriginal grid:")
for r in grid:
    print(r)

# Transpose grid
transposed = list(map(list, zip(*grid)))
print("\nTransposed grid:")
for r in transposed:
    print(r)


orig_flat = [x for row in grid for x in row if x is not None]
trans_flat = [x for row in transposed for x in row if x is not None]

transposed_data = {new: data[old] for old, new in zip(orig_flat, trans_flat)}

with open(output_path, "w") as f:
    json.dump(transposed_data, f, indent=4)

print(f"\nTransposed dictionary saved to: {output_path}")


In [ ]:

# Compare first 10 key-value pairs
print("\nFirst 10 items from original JSON:")
for i, (k, v) in enumerate(data.items()):
    if i == 10:
        break
    print(f"{i+1:2d}. {k}: {v}")

print("\nFirst 10 items from transposed JSON:")
for i, (k, v) in enumerate(transposed_data.items()):
    if i == 10:
        break
    print(f"{i+1:2d}. {k}: {v}")
